In [1]:
import json
import pandas as pd
from pathlib import Path

In [2]:
pwd

'/Users/thomasjoseph/Desktop/JMJTL/Projects/Gen_ai/GEO'

In [3]:
path = "/Users/thomasjoseph/Desktop/JMJTL/Projects/Gen_ai/GEO/geo_discovery/geo_projects/danone/aptamil/audit_raw.jsonl"

In [4]:
rows = [json.loads(l) for l in open(path, encoding="utf-8")]
df = pd.DataFrame([r for r in rows if r.get("success")])

In [5]:
df.shape

(620, 19)

In [6]:
df.head()

,query,engine,model,web_on,run_index,timestamp,success,response_text,sources,num_sources,source_type_tally,image_urls,num_images,web_search_queries,image_insights,brand_mentions,outcome,classification,intent
0,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,0,2026-09-02T06:48:58.665526+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",5,"{'other': 4, 'video_social': 1}","[[], [https://www.youtube.com/watch?v=hJczNuX5...",2,[Aptamil Pronutra formulation effectiveness in...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded
1,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,1,2026-09-02T06:49:12.309015+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",10,"{'other': 8, 'video_social': 2}","[[], [https://www.youtube.com/watch?v=tyaqo_fX...",2,[Aptamil Pronutra formulation effectiveness in...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded
2,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,2,2026-09-02T06:49:26.736799+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'bonpek.com', 'url': 'https://verte...",7,"{'other': 6, 'video_social': 1}","[[], [https://www.youtube.com/watch?v=hJczNuX5...",2,[Aptamil Pronutra infant development effective...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",brand_recommended,"{'outcome': 'brand_recommended', 'refusal_reas...",branded
3,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,3,2026-09-02T06:49:53.010825+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'nutraingredients.com', 'url': 'htt...",15,"{'other': 14, 'video_social': 1}","[[], [https://www.youtube.com/watch?v=hJczNuX5...",2,[Aptamil Pronutra infant development effective...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",brand_recommended,"{'outcome': 'brand_recommended', 'refusal_reas...",branded
4,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,4,2026-09-02T06:50:06.960163+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",9,"{'other': 8, 'video_social': 1}","[[], [https://www.youtube.com/watch?v=hJczNuX5...",2,[Aptamil Pronutra effectiveness infant develop...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded


In [11]:
def _is_first(row, target):
    bm = row.get("brand_mentions", {}) or {}
    present = [(b, v.get("first_index", -1)) for b, v in bm.items()
               if v.get("first_index", -1) >= 0]
    if not present: return False
    return min(present, key=lambda x: x[1])[0] == target

In [12]:
target_brand = "aptamil"
df["mentioned"] = df["brand_mentions"].apply(
        lambda b: b.get(target_brand, {}).get("mentioned", False))
df["first_index"] = df["brand_mentions"].apply(
        lambda b: b.get(target_brand, {}).get("first_index", -1))
df["is_first"] = df.apply(lambda r: _is_first(r, target_brand), axis=1)
df["refusal"] = df["outcome"] == "refusal"
df["sentiment"] = df["classification"].apply(
    lambda c: (c or {}).get("sentiment_toward_target") if isinstance(c, dict) else None)

In [14]:
df.head()

,query,engine,model,web_on,run_index,timestamp,success,response_text,sources,num_sources,...,image_insights,brand_mentions,outcome,classification,intent,mentioned,first_index,is_first,refusal,sentiment
0,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,0,2026-09-02T06:48:58.665526+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",5,...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded,True,0,True,False,positive
1,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,1,2026-09-02T06:49:12.309015+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",10,...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded,True,0,True,False,positive
2,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,2,2026-09-02T06:49:26.736799+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'bonpek.com', 'url': 'https://verte...",7,...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",brand_recommended,"{'outcome': 'brand_recommended', 'refusal_reas...",branded,True,0,True,False,positive
3,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,3,2026-09-02T06:49:53.010825+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'nutraingredients.com', 'url': 'htt...",15,...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",brand_recommended,"{'outcome': 'brand_recommended', 'refusal_reas...",branded,True,0,True,False,positive
4,Is Aptamil's Pronutra formulation effective fo...,gemini-2.5-flash,gemini-2.5-flash,True,4,2026-09-02T06:50:06.960163+00:00,True,Aptamil's Pronutra formulation is designed to ...,"[{'title': 'euromallusa.com', 'url': 'https://...",9,...,"[{'url': [], 'fetched': False}, {'url': ['http...","{'aptamil': {'mentioned': True, 'first_index':...",partial,"{'outcome': 'partial', 'refusal_reason': 'none...",branded,True,0,True,False,positive


In [16]:
df.columns

Index(['query', 'engine', 'model', 'web_on', 'run_index', 'timestamp',
       'success', 'response_text', 'sources', 'num_sources',
       'source_type_tally', 'image_urls', 'num_images', 'web_search_queries',
       'image_insights', 'brand_mentions', 'outcome', 'classification',
       'intent', 'mentioned', 'first_index', 'is_first', 'refusal',
       'sentiment'],
      dtype='object')

In [15]:
g = df.groupby(["engine", "intent"])["mentioned"].agg(hits="sum", runs="count").reset_index()
g

,engine,intent,hits,runs
0,azure/gpt-4o,branded,30,30
1,azure/gpt-4o,comparative,30,30
2,azure/gpt-4o,need,0,170
3,azure/gpt-4o,negative,0,50
4,azure/gpt-4o,persona,0,20
5,gemini-2.5-flash,branded,30,30
6,gemini-2.5-flash,comparative,30,30
7,gemini-2.5-flash,need,16,190
8,gemini-2.5-flash,negative,0,50
9,gemini-2.5-flash,persona,0,20


In [17]:
def _first_brand(bm):
    present = [(b, v.get("first_index",-1)) for b,v in (bm or {}).items()
               if v.get("first_index",-1) >= 0]
    return min(present, key=lambda x:x[1])[0] if present else None

In [25]:
b = "SMA"
first = df["brand_mentions"].apply(
            lambda bm: _first_brand(bm) == b).sum()
first

np.int64(128)

In [21]:
df.brand_mentions[0]

{'aptamil': {'mentioned': True, 'first_index': 0},
 'Kendamil': {'mentioned': False, 'first_index': -1},
 'HiPP Organic': {'mentioned': False, 'first_index': -1},
 'Cow & Gate': {'mentioned': False, 'first_index': -1},
 'SMA': {'mentioned': False, 'first_index': -1},
 'Enfamil': {'mentioned': False, 'first_index': -1},
 'Similac': {'mentioned': False, 'first_index': -1},
 'Aldi Mamia': {'mentioned': False, 'first_index': -1},
 'Kirkland Signature ProCare': {'mentioned': False, 'first_index': -1},
 'Bobbie Organic': {'mentioned': False, 'first_index': -1},
 'Holle Cow': {'mentioned': False, 'first_index': -1}}

In [31]:
for eng, g in df.groupby("engine"):
    for tally in g["source_type_tally"]:
        print(tally)
        print("############")
    

{'other': 3}
############
{'other': 2}
############
{'other': 4}
############
{'other': 2}
############
{'other': 2}
############
{'other': 4}
############
{'other': 2}
############
{'other': 3}
############
{'other': 3}
############
{'other': 3}
############
{}
############
{'other': 2}
############
{'other': 4}
############
{'other': 3}
############
{'other': 2}
############
{'other': 4}
############
{'other': 2}
############
{'other': 5}
############
{}
############
{}
############
{'other': 4}
############
{'other': 4}
############
{'other': 4}
############
{'other': 4}
############
{'other': 3}
############
{'other': 4}
############
{'other': 4}
############
{'other': 4}
############
{'other': 4}
############
{'other': 4}
############
{'other': 3}
############
{'other': 4}
############
{'other': 4}
############
{'other': 3}
############
{'other': 3}
############
{'other': 5}
############
{'other': 4}
############
{'other': 4}
############
{'other': 3}
############
{'other': 4}
###

In [32]:
g["source_type_tally"]

0              {'other': 4, 'video_social': 1}
1              {'other': 8, 'video_social': 2}
2              {'other': 6, 'video_social': 1}
3             {'other': 14, 'video_social': 1}
4              {'other': 8, 'video_social': 1}
                        ...                   
605                              {'other': 12}
606                              {'other': 16}
607                              {'other': 11}
608    {'other': 17, 'editorial_authority': 1}
609    {'other': 16, 'editorial_authority': 1}
Name: source_type_tally, Length: 320, dtype: object

In [37]:
sample = "\n---\n".join(df["response_text"].tolist())
sample

'Aptamil\'s Pronutra formulation is designed to support infant development by providing a blend of nutrients intended to mimic the nutritional profile of breast milk. It incorporates a scientifically formulated mix of ingredients aimed at bolstering various aspects of a baby\'s growth and health.\n\nKey components and their intended benefits for infant development include:\n*   **Essential Fatty Acids (DHA and ARA/LCPs)**: These are crucial for cognitive and visual development, playing a significant role in brain and eye health.\n*   **Prebiotics (GOS/FOS blend)**: Aptamil Pronutra contains a patented blend of galacto-oligosaccharides (GOS) and fructo-oligosaccharides (FOS) which are prebiotics. These are intended to support healthy gut bacteria, digestive health, and a robust immune system.\n*   **Vitamins and Minerals**: The formulation is enriched with vitamins A, C, and D, as well as iron and other essential nutrients. Vitamins A, C, and D are vital for immune system support, while

In [38]:
len(sample)

1800539

In [40]:
for tally in df["source_type_tally"]:
    print(tally)

{'other': 4, 'video_social': 1}
{'other': 8, 'video_social': 2}
{'other': 6, 'video_social': 1}
{'other': 14, 'video_social': 1}
{'other': 8, 'video_social': 1}
{'other': 10, 'video_social': 1}
{'other': 22, 'video_social': 1, 'editorial_authority': 4, 'community_reviews': 1}
{'other': 6, 'video_social': 1}
{'other': 8, 'video_social': 2}
{'other': 11, 'video_social': 1}
{'other': 3}
{'other': 2}
{'other': 4}
{'other': 2}
{'other': 2}
{'other': 4}
{'other': 2}
{'other': 3}
{'other': 3}
{'other': 3}
{'other': 19, 'editorial_authority': 1}
{'other': 18, 'editorial_authority': 1}
{'other': 6, 'video_social': 2}
{'other': 7}
{'other': 9}
{'other': 10}
{'other': 11}
{'other': 6}
{'other': 14, 'editorial_authority': 1}
{'other': 10, 'editorial_authority': 1}
{}
{'other': 2}
{'other': 4}
{'other': 3}
{'other': 2}
{'other': 4}
{'other': 2}
{'other': 5}
{}
{}
{'other': 6, 'video_social': 1}
{'other': 9, 'video_social': 1}
{'other': 9, 'video_social': 1}
{'other': 8, 'video_social': 1}
{'other':

In [41]:
g = df.groupby("query")["mentioned"].mean()
g

query
Aptamil vs Enfamil for infant immunity and cognitive development       1.0
Aptamil vs Kendamil: proprietary ingredient technology comparison      1.0
Aptamil vs Similac: which has better clinical credibility?             1.0
Are generic baby formulas as good as name brands?                      0.0
Are low-iron formulas better for constipation?                         0.0
Best baby formula for first-time parents                               0.0
Best baby formula for newborns                                         0.0
Best baby formula for sensitive stomach?                               0.1
Best formula for colic and gas?                                        0.1
Concerns about heavy metals in infant formula                          0.0
Does Aptamil provide a full spectrum of infant nutrition needs?        1.0
Formula for babies with dairy allergies?                               0.1
Formula for babies with reflux or spit-up?                             0.5
Formula for babies 

In [42]:
per = df.groupby(["engine","query"])["is_first"].mean().unstack(0)
per

engine,azure/gpt-4o,gemini-2.5-flash
query,,
Aptamil vs Enfamil for infant immunity and cognitive development,1.0,1.0
Aptamil vs Kendamil: proprietary ingredient technology comparison,1.0,1.0
Aptamil vs Similac: which has better clinical credibility?,1.0,1.0
Are generic baby formulas as good as name brands?,0.0,0.0
Are low-iron formulas better for constipation?,0.0,0.0
Best baby formula for first-time parents,0.0,0.0
Best baby formula for newborns,0.0,0.0
Best baby formula for sensitive stomach?,0.0,0.2
Best formula for colic and gas?,0.0,0.0


In [43]:
engines = list(per.columns)
agree_win = per[(per > 0.5).all(axis=1)].index.tolist()      # win everywhere
agree_lose = per[(per == 0).all(axis=1)].index.tolist()      # lose everywhere
diverge = per[per.max(axis=1) - per.min(axis=1) > 0.4].index.tolist()
engines, agree_win,agree_lose,diverge

(['azure/gpt-4o', 'gemini-2.5-flash'],
 ['Aptamil vs Enfamil for infant immunity and cognitive development',
  'Aptamil vs Kendamil: proprietary ingredient technology comparison',
  'Aptamil vs Similac: which has better clinical credibility?',
  'Does Aptamil provide a full spectrum of infant nutrition needs?',
  "Is Aptamil's Pronutra formulation effective for infant development?",
  "What are Aptamil's claims regarding early-life immunity?"],
 ['Are generic baby formulas as good as name brands?',
  'Are low-iron formulas better for constipation?',
  'Best baby formula for first-time parents',
  'Best baby formula for newborns',
  'Best formula for colic and gas?',
  'Concerns about heavy metals in infant formula',
  'Formula for babies with dairy allergies?',
  'Formula for babies with sensitive stomachs',
  'Formula for constipation relief?',
  'How to choose formula for a newborn?',
  'How to choose the best baby formula?',
  'Infant formula for health-conscious parents',
  'Is bab

In [45]:
import json
import re
from pathlib import Path
from collections import defaultdict

In [46]:
 comp_snippets = defaultdict(list)

In [47]:
comp_snippets

defaultdict(list, {})

In [48]:
raw_path = "/Users/thomasjoseph/Desktop/JMJTL/Projects/Gen_ai/GEO/geo_discovery/geo_projects/danone/aptamil/audit_raw.jsonl"

In [52]:
query_filter = None
comp_names = ["SMA","Enfamil","aptamil"]
for line in open(raw_path, encoding="utf-8"):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if not r.get("success"):
        continue
    if query_filter and query_filter.lower() not in r.get("query", "").lower():
        continue                                    # only the relevant query's responses
    text = r.get("response_text", "")
    #print(text)
    for sent in re.split(r'(?<=[.!?])\s+', text):
        low = sent.lower()
        for cname in comp_names:
            if cname.lower() == target_brand.lower():
                continue
            if cname.lower() in low and len(sent.strip()) > 30:
                comp_snippets[cname].append(sent.strip())

In [53]:
comp_snippets

defaultdict(list,
            {'SMA': ['**Brain and Immune Development**: Aptamil links immune system support with cognitive development, promoting the idea of "stronger immunity, smarter future" through its advanced nutritional engineering [RSIS International](https://rsisinternational.org/journals/ijriss/uploads/vol10-iss21-pg1255-1269-202608_pdf.pdf).',
              'Specialty Formulas**\n- **Kendamil**: Offers a smaller range of products, including Classic, Organic, Goat Milk, and Comfort formulas.',
              'Research and Development**\n- **Kendamil**: Focuses on traditional methods and natural ingredients, with a smaller-scale, family-run production model in the UK [Kendamil Blog](https://kendamil.com/blogs/blog/kendamil-vs-aptamil-which-follow-on-formula-is-right-for-your-baby).',
              '*   **Hydrolyzed Formulas (Hypoallergenic):** For babies with milk protein allergies or significant digestive issues, hydrolyzed formulas have proteins broken down into smaller pie

In [54]:
comp_snippets.keys()

dict_keys(['SMA', 'Enfamil'])

In [56]:
comp_snippets['SMA']

['**Brain and Immune Development**: Aptamil links immune system support with cognitive development, promoting the idea of "stronger immunity, smarter future" through its advanced nutritional engineering [RSIS International](https://rsisinternational.org/journals/ijriss/uploads/vol10-iss21-pg1255-1269-202608_pdf.pdf).',
 'Specialty Formulas**\n- **Kendamil**: Offers a smaller range of products, including Classic, Organic, Goat Milk, and Comfort formulas.',
 'Research and Development**\n- **Kendamil**: Focuses on traditional methods and natural ingredients, with a smaller-scale, family-run production model in the UK [Kendamil Blog](https://kendamil.com/blogs/blog/kendamil-vs-aptamil-which-follow-on-formula-is-right-for-your-baby).',
 '*   **Hydrolyzed Formulas (Hypoallergenic):** For babies with milk protein allergies or significant digestive issues, hydrolyzed formulas have proteins broken down into smaller pieces, making them less likely to cause allergic reactions.',
 '*   **Extensive

In [58]:
{c: list(dict.fromkeys(snips))[:8] for c, snips in comp_snippets.items()}

{'SMA': ['**Brain and Immune Development**: Aptamil links immune system support with cognitive development, promoting the idea of "stronger immunity, smarter future" through its advanced nutritional engineering [RSIS International](https://rsisinternational.org/journals/ijriss/uploads/vol10-iss21-pg1255-1269-202608_pdf.pdf).',
  'Specialty Formulas**\n- **Kendamil**: Offers a smaller range of products, including Classic, Organic, Goat Milk, and Comfort formulas.',
  'Research and Development**\n- **Kendamil**: Focuses on traditional methods and natural ingredients, with a smaller-scale, family-run production model in the UK [Kendamil Blog](https://kendamil.com/blogs/blog/kendamil-vs-aptamil-which-follow-on-formula-is-right-for-your-baby).',
  '*   **Hydrolyzed Formulas (Hypoallergenic):** For babies with milk protein allergies or significant digestive issues, hydrolyzed formulas have proteins broken down into smaller pieces, making them less likely to cause allergic reactions.',
  '*  